In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import roc_auc_score, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

customers = pd.read_csv("../data/customers.csv")

feature_cols = ["frequency", "monetary", "total_items", "unique_products",
                "recency_days", "tenure_days", "avg_order_value"]
X = customers[feature_cols]

# 1. Classification: buys again in 90 days?
X_train, X_test, y_train, y_test = train_test_split(
    X, customers["buys_again"], test_size=0.2, random_state=42, stratify=customers["buys_again"])
clf = RandomForestClassifier(n_estimators=200, random_state=42)
clf.fit(X_train, y_train)
print("Classifier AUC:", round(roc_auc_score(y_test, clf.predict_proba(X_test)[:, 1]), 3))

# 2. Regression: future spend
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X, customers["future_spend"], test_size=0.2, random_state=42)
reg = RandomForestRegressor(n_estimators=200, random_state=42)
reg.fit(Xr_train, yr_train)
pred = reg.predict(Xr_test)
print("Regressor MAE:", round(mean_absolute_error(yr_test, pred), 1))
print("Regressor R2:", round(r2_score(yr_test, pred), 3))

# 3. K-Means on RFM (log-scaled because of the heavy skew)
rfm = np.log1p(customers[["recency_days", "frequency", "monetary"]])
rfm_scaled = StandardScaler().fit_transform(rfm)
customers["segment"] = KMeans(n_clusters=4, random_state=42, n_init=10).fit_predict(rfm_scaled)
print(customers.groupby("segment")[["recency_days", "frequency", "monetary"]].mean().round(1))
print(customers["segment"].value_counts().sort_index())

# Score every customer and save for Tableau
customers["prob_buy_again"] = clf.predict_proba(X)[:, 1]
customers["predicted_spend"] = reg.predict(X)
customers.to_csv("../dashboards/customers_scored.csv", index=False)
print("Saved ../dashboards/customers_scored.csv", customers.shape)

Classifier AUC: 0.778
Regressor MAE: 390.4
Regressor R2: 0.565
         recency_days  frequency  monetary
segment                                   
0               195.2        5.2    2073.8
1                29.9       19.4   10945.6
2                31.3        2.7     736.4
3               342.2        1.4     351.1
segment
0    1622
1     884
2     647
3    2125
Name: count, dtype: int64
Saved ../dashboards/customers_scored.csv (5278, 13)
